In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("logistics.db")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

for t in tables["name"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(f"{t}: {n:,}")

shippers: 1,000
nodes: 40
truck_runs: 91,909
run_stops: 108,232
shipments: 300,004
forecasts: 69,117
holidays: 81
exceptions: 29,945


In [2]:
print(tables)

         name
0    shippers
1       nodes
2  truck_runs
3   run_stops
4   shipments
5   forecasts
6    holidays
7  exceptions


In [13]:
query = """
select * from forecasts  limit 5
"""

pd.read_sql(query,conn)

,shipper_id,week_start_date,forecast_packages
0,S0001,2025-08-18,122
1,S0001,2025-08-25,98
2,S0001,2025-09-01,129
3,S0001,2025-09-08,167
4,S0001,2025-09-15,112


In [ ]:
query = """ 
with abs as (
select a.*, b.forecast_packages 
from shipments a
left join forecasts b 
on a.shipper_id = b.shipper_id
and a.promised_date = b.pickup_date
--and a.node_id = b.node_id
)
select * from abs where forecast_packages != 0
limit 5

""" 

pd.read_sql(query,conn)

,shipment_id,shipper_id,run_id,node_id,service_level,created_date,promised_date,delivered_to_node_date,status,hold_reason,packages,pallets,forecast_packages
0,SH0129523,S0001,R040476,N-UK-07,1DD,2025-11-16,2025-11-17,2025-11-17,Delivered to Node,None,50,2,103
1,SH0173586,S0001,R053892,N-UK-01,1DD,2026-02-15,2026-02-16,2026-02-16,Delivered to Node,None,26,1,74
2,SH0184248,S0001,R057160,N-UK-01,1DD,2026-03-08,2026-03-09,2026-03-09,Delivered to Node,None,23,1,19
3,SH0195018,S0001,R060412,N-UK-01,1DD,2026-03-29,2026-03-30,2026-03-30,Delivered to Node,None,44,2,125
4,SH0217195,S0001,R067058,N-UK-01,1DD,2026-05-10,2026-05-11,2026-05-11,Delivered to Node,None,49,2,40


In [23]:
for t in tables["name"]:
    cols = pd.read_sql(f"PRAGMA table_info({t})", conn)["name"].tolist()
    print(f"{t}: {cols}")

for col, t in [("status","shipments"), ("hold_reason","shipments"),
               ("run_type","truck_runs"), ("exception_type","exceptions"),
               ("status","shippers")]:
    vals = pd.read_sql(f"SELECT DISTINCT {col} FROM {t}", conn)[col].tolist()
    print(f"{t}.{col}: {vals}")

shippers: ['shipper_id', 'name', 'country', 'city', 'onboarded_date', 'status', 'delisted_date']
nodes: ['node_id', 'node_type', 'country', 'city', 'daily_capacity_pallets']
truck_runs: ['run_id', 'run_date', 'run_type', 'carrier', 'node_id', 'planned_pickup_time', 'actual_pickup_time', 'planned_arrival_time', 'actual_arrival_time', 'pallet_capacity', 'pallets_loaded', 'cost_eur']
run_stops: ['run_id', 'shipper_id', 'stop_sequence', 'pallets_picked']
shipments: ['shipment_id', 'shipper_id', 'run_id', 'node_id', 'service_level', 'created_date', 'promised_date', 'delivered_to_node_date', 'status', 'hold_reason', 'packages', 'pallets']
forecasts: ['shipper_id', 'week_start_date', 'forecast_packages']
holidays: ['country', 'holiday_date', 'holiday_name']
exceptions: ['exception_id', 'run_id', 'shipment_id', 'exception_type', 'exception_date', 'resolved_date']
shipments.status: ['Delivered to Node', 'On Hold', 'Cancelled', 'Created', 'Picked Up', 'In Transit']
shipments.hold_reason: [nan, '